# Testing: pytest, parametrized tests, fixtures en assertions

Complete cheat sheet voor de twee testingdelen.

In [ ]:
# Maakt lokaal geinstalleerde dependencies vindbaar als pytest niet volledig globaal geinstalleerd is.
import sys
from pathlib import Path

for candidate in [Path.cwd() / '.codex_deps', Path.cwd().parent / '.codex_deps']:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))


## Code correctness

Code correctness betekent dat code doet wat de specificatie zegt, zonder onverwachte fouten.

Je kan correctheid ondersteunen met meerdere technieken:

- static typing
- verification tools
- assertions
- correctness proofs
- automated tests

In praktijk combineer je vaak meerdere technieken. Tests zijn belangrijk omdat je ze opnieuw kan uitvoeren na elke codewijziging.

## Wat maakt een test goed?

Een goede test is:

- Automated: je kan hem automatisch opnieuw uitvoeren.
- Fine grained: hij faalt liefst maar voor een reden.
- Readable: input, actie en verwachting zijn duidelijk.
- Fast: je kan hem vaak draaien tijdens development.
- Isolated: tests beinvloeden elkaar niet en volgorde mag niet uitmaken.

## Van manueel testen naar `assert`

Manueel printen zegt niet automatisch of iets fout is. Met `assert` maak je de verwachte uitkomst expliciet.

In [ ]:
def square(x):
    return x ** 2


# Manueel controleren: je moet zelf naar de output kijken.
print(square(5))
print(square(77))

# Automatisch controleren: Python stopt als de verwachting niet klopt.
assert square(5) == 25
assert square(77) == 5929

## Pytest basis

Pytest zoekt standaard naar functies waarvan de naam begint met `test_`.

Een test slaagt als de functie normaal eindigt. Een test faalt als er een exception ontstaat, vaak via een mislukte `assert`.

Typische structuur:

```python
def test_naam():
    actual = functie(input)
    expected = waarde
    assert actual == expected
```

Run tests met:

```powershell
python -m pytest
```

In [ ]:
def add(a, b):
    return a + b


def test_add_positive_numbers():
    actual = add(2, 3)
    expected = 5
    assert actual == expected


def test_add_negative_numbers():
    actual = add(-2, -3)
    expected = -5
    assert actual == expected


# In een echte testfile roept pytest deze functies aan.
# In een notebook kunnen we ze manueel uitvoeren.
test_add_positive_numbers()
test_add_negative_numbers()
print('tests passed')

## Waarom `assert` gebruiken in plaats van zelf `AssertionError` raisen?

Pytest geeft bij `assert actual == expected` veel betere foutinformatie. Het toont bijvoorbeeld welke index of waarde verschilt.

Schrijf dus liever:

```python
assert actual == expected
```

dan:

```python
if actual != expected:
    raise AssertionError()
```

## Assertion message

Je kan een extra foutboodschap toevoegen na een komma. Gebruik dit vooral als de standaard pytest-output niet duidelijk genoeg is.

In [ ]:
def is_even(number):
    return number % 2 == 0


def test_is_even():
    number = 4
    assert is_even(number), f'{number} should be even'


test_is_even()
print('test passed')

## Parametrized tests

Met `@pytest.mark.parametrize` test je dezelfde testlogica met meerdere inputs.

Waarom?

- Minder duplicatie.
- Elk geval wordt als aparte test getoond.
- Je test makkelijk grensgevallen.

Hoe?

```python
@pytest.mark.parametrize('parameter1, parameter2', [
    (waarde1, waarde2),
    (waarde1, waarde2),
])
def test_iets(parameter1, parameter2):
    ...
```

In [ ]:
import pytest


def overlapping_intervals(interval1, interval2):
    start1, end1 = interval1
    start2, end2 = interval2
    return start1 <= end2 and start2 <= end1


@pytest.mark.parametrize('interval1, interval2', [
    ((1, 5), (3, 6)),
    ((1, 5), (5, 6)),
    ((1, 10), (3, 6)),
    ((6, 8), (3, 6)),
    ((5, 7), (4, 8)),
])
def test_overlapping_intervals(interval1, interval2):
    assert overlapping_intervals(interval1, interval2), (
        f'Interval {interval1} overlaps with interval {interval2}'
    )


# Manueel enkele cases uitvoeren in deze notebook.
for interval1, interval2 in [((1, 5), (3, 6)), ((1, 5), (5, 6))]:
    test_overlapping_intervals(interval1, interval2)
print('sample parametrized cases passed')

## Floating point en `approx`

Kommagetallen zijn niet altijd exact in binary floating point.

Daarom kan dit onverwacht zijn:

```python
sum([0.1] * 10) == 1
```

Gebruik `pytest.approx` wanneer je met floats vergelijkt.

In [ ]:
from pytest import approx
from math import pi

actual = sum([0.1] * 10)
expected = 1

print(actual)
print(actual == expected)

assert approx(expected) == actual
assert approx(3.14159265) == pi
assert approx(3.14, abs=0.01) == pi
print('approx checks passed')

## Exceptions testen

Gebruik `pytest.raises` als je verwacht dat code een exception gooit.

Waarom?

- Je test niet alleen de happy path.
- Je controleert of validatie echt werkt.
- De code binnen de `with` block hou je zo klein mogelijk.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        return self.__name

    @name.setter
    def name(self, value):
        if value is None:
            raise ValueError('name cannot be None')
        self.__name = value


def test_animal_name_cannot_be_none():
    with pytest.raises(ValueError):
        Animal(None)


test_animal_name_cannot_be_none()
print('exception test passed')

## Expected values genereren

Soms kan je expected values genereren uit een bekende relatie.

Voorbeeld: als `n = x * x`, dan moet `sqrt(n)` ongeveer `x` zijn.

In [ ]:
from math import sqrt
from pytest import approx


@pytest.mark.parametrize('n, expected', [
    (x * x, x) for x in [1, 2, 3, 4, 10, 1.41421356, 76.059187]
])
def test_sqrt(n, expected):
    actual = sqrt(n)
    assert approx(expected) == actual


for n, expected in [(1, 1), (4, 2), (2, 1.41421356)]:
    test_sqrt(n, expected)
print('sqrt sample cases passed')

## Mini-overzicht

| Situatie | Pytest-techniek |
|---|---|
| Normale verwachte waarde | `assert actual == expected` |
| Zelfde test met veel inputs | `@pytest.mark.parametrize` |
| Floats vergelijken | `pytest.approx` |
| Verwachte exception | `pytest.raises` |
| Extra foutinfo nodig | `assert condition, message` |

---

# Deel 2: Setup, fixtures en assertions

## Setup voor tests

Tests hebben vaak voorbereiding nodig, bijvoorbeeld:

- objecten aanmaken
- testdata vullen
- database connectie voorbereiden
- tijdelijke bestanden maken

Als je die setup in elke test herhaalt, krijg je code-duplicatie. Daarom gebruik je setupfuncties of fixtures.

## `setup_function` en `teardown_function`

Pytest kan automatisch `setup_function` voor elke test uitvoeren. Na elke test kan `teardown_function` uitgevoerd worden.

Volgorde:

```text
setup_function()
test_1()
teardown_function()
setup_function()
test_2()
teardown_function()
```

Nadeel: als setup groot wordt, doet elke test mogelijk meer werk dan nodig.

In [ ]:
database = []


def setup_function():
    database.clear()
    database.extend(['Alice', 'Bob'])


def teardown_function():
    database.clear()


def test_database_contains_alice():
    assert 'Alice' in database


def test_database_size():
    assert len(database) == 2


# Simulatie van wat pytest per test doet.
setup_function()
test_database_contains_alice()
teardown_function()

setup_function()
test_database_size()
teardown_function()

print('setup/teardown simulation passed')

## Fixtures

Een fixture is een herbruikbaar stuk setup. Tests vragen de fixture aan via een parameter.

Waarom fixtures?

- Een test krijgt alleen de setup die hij nodig heeft.
- Minder duplicatie.
- Fixtures kunnen zelf dependencies hebben.
- Met `yield` kan je setup en teardown in een fixture combineren.

In [ ]:
import pytest


@pytest.fixture
def sample_students():
    return [
        {'name': 'Alice', 'score': 85},
        {'name': 'Bob', 'score': 78},
        {'name': 'Charlie', 'score': 92},
    ]


def average_score(students):
    return sum(student['score'] for student in students) / len(students)


def test_average_score(sample_students):
    assert average_score(sample_students) == pytest.approx(85)


# In pytest wordt sample_students automatisch aangeroepen.
# In een notebook voeren we de kernlogica manueel uit.
students = [
    {'name': 'Alice', 'score': 85},
    {'name': 'Bob', 'score': 78},
    {'name': 'Charlie', 'score': 92},
]
assert average_score(students) == pytest.approx(85)
print('fixture example logic passed')

## Fixture met `yield`

Alles voor `yield` is setup. Alles na `yield` is teardown.

Dit is handig voor resources die je achteraf moet opruimen.

In [ ]:
import pytest


@pytest.fixture
def opened_resource():
    resource = {'open': True, 'items': []}
    yield resource
    resource['open'] = False


def add_item(resource, item):
    if not resource['open']:
        raise ValueError('resource is closed')
    resource['items'].append(item)


def test_add_item(opened_resource):
    add_item(opened_resource, 'book')
    assert opened_resource['items'] == ['book']


# Notebook-simulatie van setup en teardown.
resource = {'open': True, 'items': []}
add_item(resource, 'book')
assert resource['items'] == ['book']
resource['open'] = False
print('yield fixture idea passed')

## Fixture dependencies

Een fixture kan een andere fixture als parameter vragen. Pytest bouwt dan automatisch de juiste setup-volgorde.

In [ ]:
import pytest


@pytest.fixture
def user():
    return {'name': 'Vince'}


@pytest.fixture
def authenticated_user(user):
    user['authenticated'] = True
    return user


def test_authenticated_user(authenticated_user):
    assert authenticated_user['authenticated'] is True
    assert authenticated_user['name'] == 'Vince'


# Notebook-simulatie.
user_data = {'name': 'Vince'}
user_data['authenticated'] = True
assert user_data['authenticated'] is True
print('fixture dependency idea passed')

## Assertions in gewone code

`assert` kan ook in gewone code gebruikt worden als self-check.

Voorbeeld: een eigen `max` functie kan na berekening controleren of:

- het resultaat effectief in de lijst zit
- geen enkel element groter is dan het resultaat

Let op: gebruik `assert` niet voor normale inputvalidatie in productiecode. Daarvoor gebruik je expliciete exceptions.

In [ ]:
def max_checked(numbers):
    if len(numbers) == 0:
        raise ValueError('numbers cannot be empty')

    result = numbers[0]
    for number in numbers:
        if number > result:
            result = number

    assert result in numbers
    assert all(number <= result for number in numbers)
    return result


print(max_checked([1, 5, 7, 3, 8, 2, 9, 5, 1]))

## Debug vs release / optimized mode

In Python kan je optimized mode starten met `-O`.

Belangrijk verschil: asserts worden dan genegeerd.

```powershell
python script.py      # asserts actief
python -O script.py   # asserts genegeerd
```

Daarom gebruik je `assert` niet voor verplichte runtime-validatie van gebruikersinput. Gebruik dan `raise ValueError`, `raise TypeError`, enz.

In [ ]:
def set_age(age):
    # Minder geschikt voor echte inputvalidatie, want assert kan genegeerd worden met python -O.
    assert age >= 0, 'age cannot be negative'
    return age


def set_age_safe(age):
    # Beter voor productievalidatie.
    if age < 0:
        raise ValueError('age cannot be negative')
    return age


print(set_age(10))
print(set_age_safe(10))

## Mini-overzicht

| Situatie | Aanpak |
|---|---|
| Elke test dezelfde setup | `setup_function` |
| Alleen nodige setup per test | `@pytest.fixture` |
| Setup + teardown samen | fixture met `yield` |
| Fixture heeft andere setup nodig | fixture dependency via parameter |
| Self-check tijdens development | `assert` |
| Productievalidatie | expliciete exception |